<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 1 · Python Programming Foundations for Data Science
### Notebook 04 · Text Methods and Titanic

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
Chapter 4 uses Titanic as a string-heavy warmup. The point is not analysis
yet. The point is to get comfortable with raw text, cleaning, and counting.

### Goals
- Open a local Titanic CSV or download a copy if needed.
- Inspect the raw rows as text.
- Clean a few names and count embarkation ports.

This cell continues the worked example for the current section.


This cell prepares the notebook for local or Colab execution. It finds the book root, clones the companion repo when needed, and makes the working directory predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "1_data"  # book folder to locate locally or after clone
REPO_URL = "https://github.com/yhilpisch/tdscode"  # companion repo with notebooks, data, and code

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = None  # filled once a matching book directory is found
for candidate in [cwd, *cwd.parents]:  # search upward for the book root
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():  # local book root
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():  # repo/book layout
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")  # Colab clone location
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)  # fetch the companion repo once
    BOOK_ROOT = repo_root / BOOK_NAME  # book folder inside the clone

os.chdir(BOOK_ROOT)  # keep relative paths anchored on the book root
if str(BOOK_ROOT) not in sys.path:  # allow src/ imports
    sys.path.insert(0, str(BOOK_ROOT))  # allow src/ imports

code_dir = BOOK_ROOT / "code"  # helper scripts live here
if code_dir.exists() and str(code_dir) not in sys.path:  # make helper scripts importable
    sys.path.insert(0, str(code_dir))

requirements = BOOK_ROOT / "requirements.txt"  # install only when present
if requirements.exists() and "google.colab" in sys.modules:  # keep Colab in sync with the book
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)  # keep Colab in sync with the book

print("Book root:", BOOK_ROOT)


In [ ]:
import csv
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = 'https://hilpisch.com/Titanic.csv'
LOCAL_CANDIDATES = [
    BOOK_ROOT / 'data' / 'Titanic.csv',
    Path('../2_data/data') / 'Titanic.csv',
    Path('../../2_data/data') / 'Titanic.csv',
]

def resolve_data_path() -> Path:
    for candidate in LOCAL_CANDIDATES:
        if candidate.exists():
            return candidate
    target = LOCAL_CANDIDATES[0]
    target.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(DATA_URL, target)
    return target

def preview_rows(path: Path, limit: int = 5) -> list[str]:
    rows = []
    with path.open(newline='', encoding='utf-8') as handle:
        reader = csv.reader(handle)
        for _ in range(limit):
            try:
                row = next(reader)
            except StopIteration:
                break
            rows.append(', '.join(row))
    return rows


This cell continues the worked example for the current section.


In [ ]:
def clean_name(name: str) -> str:
    return ' '.join(name.strip().split()).title()

def count_embarked(path: Path) -> dict[str, int]:
    counts = {}
    with path.open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            port = (row.get('Embarked') or '').strip().upper() or '?'
            counts[port] = counts.get(port, 0) + 1
    return counts

data_path = resolve_data_path()
raw_preview = preview_rows(data_path)
counts = count_embarked(data_path)

print(f'Using Titanic CSV at {data_path.resolve()}')
print('Raw preview:')
for line in raw_preview:
    print(f'  {line}')
print('Embarkation counts:')
for port, count in sorted(counts.items()):
    print(f'  {port}: {count}')
print('Example cleaned names:')
with data_path.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    for index, row in enumerate(reader):
        print(f"  {clean_name(row.get('Name') or '')}")
        if index >= 4:
            break


This cell continues the worked example for the current section.


In [ ]:
from pathlib import Path

artifact_path = BOOK_ROOT / 'artifacts' / 'titanic_strings_summary.txt'
artifact_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_names = []
with data_path.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    for index, row in enumerate(reader):
        cleaned_names.append(clean_name(row.get('Name') or ''))
        if index >= 9:
            break

lines = ['Titanic strings summary', '', 'Raw preview:']
lines.extend(f'  {line}' for line in raw_preview)
lines.extend(['', 'Embarkation counts:'])
lines.extend(f'  {port}: {count}' for port, count in sorted(counts.items()))
lines.extend(['', 'Cleaned names:'])
lines.extend(f'  {name}' for name in cleaned_names)
artifact_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(artifact_path.resolve())


### Reflection
- Why does the raw CSV preview matter before any analysis?
- What is the difference between text cleaning and data analysis?
- Which string operation would you reuse in a later module?

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
